# KV Cache 实测：vLLM Prefix Caching on Colab (T4)

配套博客《KV Cache 详解：从 PagedAttention 到 Prefix Caching》。

这个 notebook 在 **Colab 免费 T4 GPU** 上跑**真实** vLLM，量出 Prefix Caching 对 **TTFT** 的影响，用一手延迟数据印证文章里的仿真结论：

- 实验 1：同一长前缀，**冷（首次）vs 热（命中缓存）**；并对比 Prefix Caching **开 / 关**。
- 实验 2：同样的内容，易变字段放在**开头 vs 结尾**，TTFT 天壤之别。

**用法**：菜单 `Runtime → Change runtime type → T4 GPU`，然后 `Runtime → Run all`（首次装 vLLM 约几分钟）。跑完把打印结果贴回给我，或下载生成的 `vllm-prefix-caching-ttft.png`。

In [ ]:
!nvidia-smi

In [ ]:
# vLLM 安装较大，约需几分钟；若最新版在 T4 上有问题，可改为 !pip install -q "vllm==0.6.6.post1"
!pip install -q vllm

In [ ]:
import time, statistics, gc, random
import torch
from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # T4 友好；显存紧张可换 "Qwen/Qwen2.5-0.5B-Instruct"
SP = SamplingParams(max_tokens=1, temperature=0.0)  # 只生成 1 个 token ≈ 只测 prefill / TTFT

print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (先切到 T4)")


def make_llm(prefix_caching):
    return LLM(
        model=MODEL,
        enable_prefix_caching=prefix_caching,
        gpu_memory_utilization=0.85,
        max_model_len=4096,
        dtype="half",
        enforce_eager=True,  # T4 上更稳
    )


def free(llm):
    try:
        del llm
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()


def gen_ms(llm, prompt):
    t0 = time.perf_counter()
    llm.generate([prompt], SP, use_tqdm=False)
    return (time.perf_counter() - t0) * 1000.0


def cold_warm(llm, prompt, runs=5):
    llm.generate(["warmup cuda kernels"], SP, use_tqdm=False)  # 预热，排除首次编译噪声
    cold = gen_ms(llm, prompt)                                  # 该前缀首次出现 = 冷
    warm = statistics.median([gen_ms(llm, prompt) for _ in range(runs)])  # 重复 = 命中前缀缓存
    return cold, warm

In [ ]:
SHARED = "You are a meticulous assistant. Follow the rules carefully. " * 180  # ~1k+ tokens
QUESTION = "\n\nUser: Summarize the above policy in one sentence.\nAssistant:"
shared_prompt = SHARED + QUESTION

llm_on = make_llm(True)

# 实验 1（ON）：同一长前缀 冷 vs 热
cold_on, warm_on = cold_warm(llm_on, shared_prompt)
print(f"[ON]  cold={cold_on:.1f}ms  warm={warm_on:.1f}ms  speedup={cold_on / warm_on:.1f}x")

# 实验 2：易变字段放头 vs 放尾（各测多条不同 volatile 值的“热”请求）
def volatile(i):
    return f"[req-{i}-{random.randint(0, 10**9)}] "

llm_on.generate([shared_prompt], SP, use_tqdm=False)  # 先把共享前缀预热进缓存
head_ms, tail_ms = [], []
for i in range(8):
    v = volatile(i)
    head_ms.append(gen_ms(llm_on, v + shared_prompt))  # 变量在最前 -> 首块每次变 -> 不命中
    tail_ms.append(gen_ms(llm_on, shared_prompt + v))  # 变量在最后 -> 共享前缀命中
head_ttft = statistics.median(head_ms)
tail_ttft = statistics.median(tail_ms)
print(f"[head volatile] TTFT={head_ttft:.1f}ms    [tail volatile] TTFT={tail_ttft:.1f}ms")

In [ ]:
free(llm_on)
llm_off = make_llm(False)
cold_off, warm_off = cold_warm(llm_off, shared_prompt)
print(f"[OFF] cold={cold_off:.1f}ms  warm={warm_off:.1f}ms  speedup={cold_off / warm_off:.1f}x (期望≈1)")
free(llm_off)

In [ ]:
import matplotlib.pyplot as plt

vals = [cold_on, warm_on, cold_off, warm_off]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.bar(["ON\ncold", "ON\nwarm", "OFF\ncold", "OFF\nwarm"], vals,
       color=["#dc2626", "#16a34a", "#dc2626", "#9ca3af"])
a1.set_ylabel("TTFT (ms, max_tokens=1)")
a1.set_title("Prefix Caching: cold vs warm")
for i, v in enumerate(vals):
    a1.text(i, v, f"{v:.0f}", ha="center", va="bottom")

vals2 = [head_ttft, tail_ttft]
a2.bar(["volatile\nat HEAD", "volatile\nat TAIL"], vals2, color=["#ea7317", "#16a34a"])
a2.set_ylabel("TTFT (ms)")
a2.set_title("Same content, ordering decides latency")
for i, v in enumerate(vals2):
    a2.text(i, v, f"{v:.0f}", ha="center", va="bottom")

fig.suptitle(f"vLLM real benchmark on T4 - {MODEL}", fontweight="bold")
fig.tight_layout()
fig.savefig("vllm-prefix-caching-ttft.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved vllm-prefix-caching-ttft.png")

try:
    from google.colab import files
    files.download("vllm-prefix-caching-ttft.png")
except Exception as e:
    print("skip download:", e)

## 把结果贴回来

跑完后，请把上面几个单元格的打印（`[ON] cold=… warm=…`、`[OFF] …`、`[head/tail] …`）复制给我，或下载 `vllm-prefix-caching-ttft.png`。我会把这组**真实 T4 数据**并入文章，与仿真结果并列对照。

> 说明：`max_tokens=1` 让 `generate` 的耗时 ≈ prefill 时间，作为 TTFT 的近似。若 T4 显存吃紧，把 `MODEL` 换成 `Qwen/Qwen2.5-0.5B-Instruct` 或调低 `gpu_memory_utilization`。